<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB03 &mdash; Transaction Loading</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Load every transaction across all portfolios: equities, fixed income, OTC, cash, multi asset, and the S&P 500, AI/Tech, Blockchain and Global Aggregate iShares portfolios.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; **NB03** &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB03: Transaction Loading

Loads all transactions: equities, fixed income, OTC, cash, multi asset, S&P 500, AI, blockchain, and global aggregate bonds.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp

pd.set_option("display.max_columns", None)

SCOPE = 'ibor-training-v9'
DATA_DIR = 'data'

def get_factory(app='lusid'):
    secrets_path = "secrets.json"
    with open(secrets_path) as f:
        secrets = json.load(f)
    api_section = secrets.get("api", {})
    pat = api_section.get("accessToken")
    module = __import__(app)
    if pat:
        config_loaders = [module.extensions.ArgsConfigurationLoader(
            api_url=api_section.get("lusidUrl", ""), access_token=pat)]
    else:
        config_loaders = [module.extensions.SecretsFileConfigurationLoader(secrets_path)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    secrets_path = "secrets.json"
    with open(secrets_path) as f:
        secrets = json.load(f)
    api_section = secrets.get("api", {})
    pat = api_section.get("accessToken")
    if pat:
        lumi_url = api_section.get("lusidUrl", "").replace("/api", "/honeycomb")
        return lp.get_client(access_token=pat, api_url=lumi_url)
    return lp.get_client(api_secrets_file=secrets_path)

lusid_factory = get_factory('lusid')
lumi = get_lumi()
print('LUSID + Luminesce initialised')

transaction_portfolios_api = lusid_factory.build(lu.TransactionPortfoliosApi)
portfolios_api = lusid_factory.build(lu.PortfoliosApi)
portfolio_groups_api = lusid_factory.build(lu.PortfolioGroupsApi)
cas_api = lusid_factory.build(lu.CorporateActionSourcesApi)
transaction_portfolios_api = lusid_factory.build(lu.TransactionPortfoliosApi)
print('Ready')

---
## Transaction Loader Helper

In [ ]:
def load_equity_transactions(csv_path, portfolio_code):
    df = pd.read_csv(csv_path)
    txns = []
    for _, row in df.iterrows():
        trade_date = datetime.fromisoformat(row['TradeDate']).replace(tzinfo=timezone.utc)
        price = float(row['Price'])
        units = float(row['Units'])
        total = units * price
        
        props = {}
        for prop, col in [("Strategy", "Strategy"), ("CustodianAccount", "CustodianAccount"), ("Broker", "Broker"),
                        ("SettlementDate", "SettlementDate"), ("ValueDate", "ValueDate"),
                        ("TradeTime", "TradeTime"), ("ExecutionVenue", "ExecutionVenue"),
                        ("OrderId", "OrderId"), ("AllocationId", "AllocationId"),
                        ("CounterpartyId", "CounterpartyId"), ("SettlementInstructionType", "SettlementInstructionType"),
                        ("CommissionAmount", "CommissionAmount"), ("CommissionCurrency", "CommissionCurrency"),
                        ("TaxAmount", "TaxAmount"), ("AccruedInterest", "AccruedInterest"),
                        ("SettlementStatus", "SettlementStatus"), ("ConfirmationStatus", "ConfirmationStatus"),
                        ("TradeSource", "TradeSource"), ("PlaceOfTrade", "PlaceOfTrade"),
                        ("PlaceOfClearing", "PlaceOfClearing"), ("OriginalOrderType", "OriginalOrderType")]:
            if pd.notna(row.get(col)) and str(row[col]).strip():
                props[f"Transaction/{SCOPE}/{prop}"] = lm.PerpetualProperty(
                    key=f"Transaction/{SCOPE}/{prop}",
                    value=lm.PropertyValue(label_value=str(row[col]).strip()))
        
        txns.append(lm.TransactionRequest(
            transaction_id=row['TransactionId'], type=row['Type'],
            instrument_identifiers={"Instrument/default/ClientInternal": row['ClientInternal']},
            transaction_date=trade_date.isoformat(),
            settlement_date=(trade_date + timedelta(days=2)).isoformat(),
            units=units,
            transaction_price=lm.TransactionPrice(price=price, type="Price"),
            total_consideration=lm.CurrencyAndAmount(amount=total, currency=row['Currency']),
            transaction_currency=row['Currency'],
            properties=props if props else None
        ))
    
    transaction_portfolios_api.upsert_transactions(scope=SCOPE, code=portfolio_code, transaction_request=txns)
    return len(txns)

print("Helper function ready")

---
## Load Original Portfolio Transactions

In [ ]:
# Equity portfolio
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_equity.csv", "IBOR-EQ")
print(f"  IBOR-EQ: {n} equity transactions")

# Fixed income portfolio
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_fi.csv", "IBOR-FI")
print(f"  IBOR-FI: {n} FI transactions")

# Multi asset portfolio
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_ma.csv", "IBOR-MA")
print(f"  IBOR-MA: {n} multi asset transactions")

---
## Load OTC Transactions (IRS + Term Deposit)
Note: Term deposit booked with units=1 (one contract of $5M size).

In [ ]:
df_otc = pd.read_csv(f"{DATA_DIR}/ibor_transactions_otc.csv")
for _, row in df_otc.iterrows():
    trade_date = datetime.fromisoformat(row['TradeDate']).replace(tzinfo=timezone.utc)
    units = float(row['Units'])
    price = float(row['Price'])
    
    # For TD, total consideration = contract size ($5M)
    if 'TD' in row['ClientInternal']:
        total = 5000000
    else:
        total = 0
    
    props = {}
    if pd.notna(row.get('Strategy')) and str(row['Strategy']).strip():
        props[f"Transaction/{SCOPE}/Strategy"] = lm.PerpetualProperty(
            key=f"Transaction/{SCOPE}/Strategy",
            value=lm.PropertyValue(label_value=str(row['Strategy']).strip()))
    
    txn = [lm.TransactionRequest(
        transaction_id=row['TransactionId'], type=row['Type'],
        instrument_identifiers={"Instrument/default/ClientInternal": row['ClientInternal']},
        transaction_date=trade_date.isoformat(), settlement_date=trade_date.isoformat(),
        units=units,
        transaction_price=lm.TransactionPrice(price=price, type="Price"),
        total_consideration=lm.CurrencyAndAmount(amount=total, currency=row['Currency']),
        transaction_currency=row['Currency'],
        properties=props if props else None
    )]
    transaction_portfolios_api.upsert_transactions(scope=SCOPE, code=row['Portfolio'], transaction_request=txn)
    print(f"  {row['TransactionId']}: {row['ClientInternal']} units={units}")

---
## Load New Portfolio Transactions

In [ ]:
# S&P 500
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_sp500.csv", "IBOR-SP500")
print(f"  IBOR-SP500: {n} transactions")

# AI Innovation
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_aitech.csv", "IBOR-AITECH")
print(f"  IBOR-AITECH: {n} transactions")

# Blockchain Technology
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_blkc.csv", "IBOR-BLKC")
print(f"  IBOR-BLKC: {n} transactions")

# Global Aggregate Bond
n = load_equity_transactions(f"{DATA_DIR}/ibor_transactions_gagg.csv", "IBOR-GAGG")
print(f"  IBOR-GAGG: {n} transactions")

---
## Load Cash Funding

In [ ]:
df_cash = pd.read_csv(f"{DATA_DIR}/ibor_transactions_cash.csv")
for _, row in df_cash.iterrows():
    trade_date = datetime.fromisoformat(str(row['TradeDate']).strip()).replace(tzinfo=timezone.utc)
    amount = float(row['Amount'])
    
    props = {}
    if pd.notna(row.get('Strategy')) and str(row['Strategy']).strip():
        props[f"Transaction/{SCOPE}/Strategy"] = lm.PerpetualProperty(
            key=f"Transaction/{SCOPE}/Strategy",
            value=lm.PropertyValue(label_value=str(row['Strategy']).strip()))
    if pd.notna(row.get('CustodianAccount')) and str(row['CustodianAccount']).strip():
        props[f"Transaction/{SCOPE}/CustodianAccount"] = lm.PerpetualProperty(
            key=f"Transaction/{SCOPE}/CustodianAccount",
            value=lm.PropertyValue(label_value=str(row['CustodianAccount']).strip()))
    
    txn = [lm.TransactionRequest(
        transaction_id=row['TransactionId'], type=row['Type'],
        instrument_identifiers={"Instrument/default/Currency": row['Currency']},
        transaction_date=trade_date.isoformat(), settlement_date=trade_date.isoformat(),
        units=amount,
        transaction_price=lm.TransactionPrice(price=1, type="Price"),
        total_consideration=lm.CurrencyAndAmount(amount=amount, currency=row['Currency']),
        transaction_currency=row['Currency'],
        properties=props if props else None
    )]
    transaction_portfolios_api.upsert_transactions(scope=SCOPE, code=row['Portfolio'], transaction_request=txn)
    print(f"  {row['Portfolio']}: {row['Currency']} {amount:,.0f}")

---
## Verification

In [ ]:
query = f"SELECT PortfolioCode, COUNT(*) as TxnCount FROM Lusid.Portfolio.Txn WHERE PortfolioScope = '{SCOPE}' GROUP BY PortfolioCode LIMIT 20"
try:
    df = lumi.run(query, quiet=True)
    print("Transactions per portfolio:")
    display(df)
except Exception as e:
    print(f"Query error: {e}")

print("\nNB03 Complete. Proceed to NB04: Holdings & Position Management.")